# Feature Selection

**Objetivo**: Validación y selección de *features*, evitando *data leakage* y preparando el conjunto **Top-60** para el entrenamiento del modelo de predicción del MERVAL.


## Tabla de contenidos

1. [Setup & Config](#setup-config)  
2. [Carga de dataset de ingeniería](#carga-dataset)  
3. [Creación del *target* `merval_apertura_+1`](#creacion-target)  
4. [Partición temporal y *helpers*](#tscv-helpers)  
5. [C-Serie: Filtros de validación](#c-serie)  
   - C1. *Missing values*  
   - C2. *Low variance*  
   - C3. *Clustering de correlación* + *Mutual Information*  
6. [Proceso en paralelo: Relevancias](#paralelo-relevancias)  
   - C4. XGBoost  
   - C5. Boruta  
   - C6. SHAP  
7. [Selecciones Top-K y Top-60 final](#selecciones)  
8. [Export de artefactos](#export)  
9. [Appendix: Utils](#appendix)


### Config  <a id='setup-config'></a>

- Configurar rutas de entrada/salida.
- Definir prefijos/columnas *mandatorias* (todas las `merval_*` salvo el *target*).
- Asegurar semillas reproducibles.
- Evitar *data leakage*: toda métrica basada en *target* se estima **sólo** en *folds de entrenamiento* con `TimeSeriesSplit`.


In [1]:
#!pip install boruta

In [2]:
import os
import re
import json
import math
import warnings
from pathlib import Path
from collections import defaultdict, Counter
from datetime import datetime

import numpy as np
import pandas as pd

from sklearn.model_selection import TimeSeriesSplit
from sklearn.feature_selection import mutual_info_regression
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler

# Modelos / métodos
try:
    from xgboost import XGBRegressor
    _HAS_XGB = True
except Exception as _e:
    _HAS_XGB = False
    print("[WARN] xgboost no disponible. Instalar con: pip install xgboost")

try:
    from boruta import BorutaPy
    from sklearn.ensemble import RandomForestRegressor
    _HAS_BORUTA = True
except Exception as _e:
    _HAS_BORUTA = False
    print("[WARN] boruta no disponible. Instalar con: pip install boruta")

try:
    import shap
    _HAS_SHAP = True
except Exception as _e:
    _HAS_SHAP = False
    print("[WARN] shap no disponible. Instalar con: pip install shap")

DATASET_NAME = "dataset_v2"

# Ruta del dataset original :
DATA_PATH = f"""../inputs/{DATASET_NAME}.csv"""

# Ruta del dataset con features generadas
INPUT_DATA_FILE = f"""../inputs/{DATASET_NAME}_with_fe.csv"""

# Índice temporal
TIME_COL_CANDIDATES = ["date", "fecha"]

# Semillas
SEED = 31
rng = np.random.default_rng(SEED)

# Prefijo de *features* mandatorias (no se dropean en C1–C3)
MANDATORY = False
MANDATORY_PREFIX = "merval_"
MANDATORY_EXACTS = []

# Parámetros de selección
MAX_MISSING_FRAC     = 0.01   # C1
CONST_DOMINANCE_THR  = 0.99   # C2
CORR_CLUSTER_THR     = 0.95   # C3 (|corr| >= 0.95)

# CV temporal
N_SPLITS = 5
TSCV = TimeSeriesSplit(n_splits=N_SPLITS)

# Top-K para cada método
TOPK_PER_METHOD = 30

# Nombre de la columna base de apertura que usaremos para construir el target
TARGET_BASE = "merval_apertura"

# Columnas a dropear
DROP_COL = ["merval_cierre", "merval_maximo", "merval_minimo"]

# Utils de log
def _stamp(msg):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}")


### Input - Carga de datos  <a id='carga-dataset'></a>

- Buscar automáticamente el dataset exportado por `01_feature_engineering.ipynb`.
- Si no se encuentra, especificar `INPUT_DATA_FILE` en la celda de *Config*.


In [3]:
import nbformat as nbf
import pandas as pd

def _try_read_table(path: Path):
    if not path.exists():
        return None
    if path.suffix.lower() in (".parquet", ".pq"):
        return pd.read_parquet(path)
    if path.suffix.lower() in (".feather", ".ft"):
        return pd.read_feather(path)
    if path.suffix.lower() in (".csv",):
        return pd.read_csv(path)
    return None

df = _try_read_table(Path(INPUT_DATA_FILE))

if df is None:
    raise FileNotFoundError("No se encontró el dataset. Edite INPUT_DATA_FILE en la celda de Config para apuntar al archivo correcto.")

# Normalizar índice temporal si existe
def _to_datetime_index(idx):
    if isinstance(idx, pd.DatetimeIndex):
        return idx
    try:
        return pd.to_datetime(idx)
    except Exception:
        return pd.Index(idx)

# Detectar columna temporal para setear como índice (si no lo está)
if not isinstance(df.index, pd.DatetimeIndex):
    for cand in TIME_COL_CANDIDATES:
        if cand in df.columns:
            df[cand] = pd.to_datetime(df[cand], errors="coerce")
            if df[cand].notna().sum() >= int(0.9 * len(df)):
                df = df.set_index(cand).sort_index()
                break

df.index = _to_datetime_index(df.index)
df = df.sort_index()

df = df.drop(columns=DROP_COL, errors="ignore")

_stamp(f"df shape: {df.shape}")
df.head(3)


[19:47:38] df shape: (645, 354)


,emae_original,emae_desestacionalizado,emae_tendencia_ciclo,badlar,tc_mayorista,tc_minorista,base_monetaria,m1,m2,m2_transaccional,...,m1_std20,m2_std20,m2_transaccional_std20,prestamos_privados_ars_std20,tasa_prestamos_personales_std20,inflacion_std20,merval_apertura_std20,merval_maximo_std20,merval_minimo_std20,merval_cierre_std20
fecha,,,,,,,,,,,,,,,,,,,,,
2023-01-02,143.03019,149.687389,149.346903,67.750,178.14,185.36,5200397.0,7515263.0,12877548.0,9643075.300,...,307038.897917,678183.891400,484732.364200,143759.870873,1.496286,0.201246,13893.848386,15002.657340,13787.978797,15230.421698
2023-01-03,143.03019,149.687389,149.346903,68.375,178.43,185.99,5219463.0,7495631.0,12720840.0,9544023.507,...,293412.209110,664671.017505,487386.735245,151092.723305,1.576524,0.277014,15241.478902,15858.903720,14478.830151,15603.588796
2023-01-04,143.03019,149.687389,149.346903,69.375,178.65,186.22,5247080.0,7521155.0,12670683.0,9587667.872,...,287180.856427,638620.457839,489511.336256,151481.545702,1.728303,0.329713,15612.444971,16106.798439,14746.317186,15716.626270


## 3) Creación del *target* `merval_apertura_+1`  <a id='creacion-target'></a>

- Detectar la columna de apertura del MERVAL (`merval_apertura` o similar).
- Construir el *target* como valor **+1 paso** (*shift* hacia el futuro): `target = apertura.shift(-1)`.
- Evitar *leakage*: el *shift* se aplica **después** de alinear el índice temporal.


In [4]:
# Crear target +1
TARGET_COL = "merval_apertura_+1"
df[TARGET_COL] = df[TARGET_BASE].shift(-1)

# Quitar la última fila sin target (por el shift)
df = df.iloc[:-1, :].copy()
_stamp(f"Target creado a partir de '{TARGET_BASE}'. df shape: {df.shape}")

# Separar X, y
y = df[TARGET_COL].copy()
X = df.drop(columns=[TARGET_COL]).copy()

Stamp = _stamp
Stamp(f"X shape: {X.shape} | y shape: {y.shape}")

[19:47:38] Target creado a partir de 'merval_apertura'. df shape: (644, 355)
[19:47:38] X shape: (644, 354) | y shape: (644,)


### Partición temporal y *helpers*  <a id='tscv-helpers'></a>

- `TimeSeriesSplit` con `n_splits=5` por defecto.
- Funciones auxiliares para:
  - Cálculo de *mutual information* promedio en *folds* (entrenamiento).
  - Importancias XGBoost promedio en *folds* (entrenamiento).
  - SHAP valores absolutos medios en *folds* (entrenamiento).  
  - Boruta (wrapper RF) sobre *fold* final (opción reproducible y eficiente).


In [5]:
def is_mandatory(col: str) -> bool:
    if col in MANDATORY_EXACTS:
        return True
    return col.startswith(MANDATORY_PREFIX)

def cv_mutual_info(X: pd.DataFrame, y: pd.Series) -> pd.Series:
    mi_vals = np.zeros(X.shape[1], dtype=float)
    counts = np.zeros(X.shape[1], dtype=float)
    for fold, (tr, va) in enumerate(TSCV.split(X)):
        Xtr, ytr = X.iloc[tr], y.iloc[tr]
        # StandardScaler opcional (no requerido para MI)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            mi = mutual_info_regression(Xtr.fillna(0), ytr.values, random_state=SEED)
        mi_vals += np.nan_to_num(mi)
        counts += 1
    mi_vals = np.divide(mi_vals, counts, out=np.zeros_like(mi_vals), where=counts>0)
    return pd.Series(mi_vals, index=X.columns).sort_values(ascending=False)

def cv_xgb_importance(X: pd.DataFrame, y: pd.Series) -> pd.Series:
    if not _HAS_XGB:
        return pd.Series(0.0, index=X.columns)
    importances = np.zeros(X.shape[1], dtype=float)
    counts = np.zeros(X.shape[1], dtype=float)
    for fold, (tr, va) in enumerate(TSCV.split(X)):
        Xtr, ytr = X.iloc[tr], y.iloc[tr]
        model = XGBRegressor(
            n_estimators=350,
            max_depth=5,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=SEED,
            n_jobs=-1,
            tree_method="hist",
        )
        model.fit(Xtr.fillna(0), ytr.values)
        try:
            imp = model.feature_importances_
        except Exception:
            imp = np.zeros(X.shape[1], dtype=float)
        importances += np.nan_to_num(imp)
        counts += 1
    importances = np.divide(importances, counts, out=np.zeros_like(importances), where=counts>0)
    return pd.Series(importances, index=X.columns).sort_values(ascending=False)

def fold_shap_importance(X: pd.DataFrame, y: pd.Series) -> pd.Series:
    """SHAP mean(|value|) promedio en folds de entrenamiento con XGBRegressor.
    Si SHAP no está disponible, devuelve ceros.
    """
    if not (_HAS_XGB and _HAS_SHAP):
        return pd.Series(0.0, index=X.columns)

    shap_vals_accum = np.zeros(X.shape[1], dtype=float)
    counts = np.zeros(X.shape[1], dtype=float)

    for fold, (tr, va) in enumerate(TSCV.split(X)):
        Xtr, ytr = X.iloc[tr], y.iloc[tr]
        model = XGBRegressor(
            n_estimators=250,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=SEED + fold,
            n_jobs=-1,
            tree_method="hist",
        )
        model.fit(Xtr.fillna(0), ytr.values)

        # SHAP sobre una muestra del set de entrenamiento (para performance)
        sample_n = min(2000, len(Xtr))
        X_sample = Xtr.sample(n=sample_n, random_state=SEED).fillna(0)

        try:
            explainer = shap.TreeExplainer(model)
            shap_vals = explainer.shap_values(X_sample)
            shap_mean_abs = np.mean(np.abs(shap_vals), axis=0)
        except Exception:
            shap_mean_abs = np.zeros(X.shape[1], dtype=float)

        shap_vals_accum += shap_mean_abs
        counts += 1

    shap_vals_accum = np.divide(shap_vals_accum, counts, out=np.zeros_like(shap_vals_accum), where=counts>0)
    return pd.Series(shap_vals_accum, index=X.columns).sort_values(ascending=False)

def boruta_selection(X: pd.DataFrame, y: pd.Series) -> pd.Series:
    """Corre Boruta (RF) en el último fold de entrenamiento y devuelve un puntaje normalizado por z-score
    donde *selected* recibe +1, *tentative* +0.5, *rejected* 0. Si Boruta no está disponible: ceros.
    """
    if not _HAS_BORUTA:
        return pd.Series(0.0, index=X.columns)

    # Tomar el último fold (más reciente) para un *snapshot* representativo
    folds = list(TSCV.split(X))
    tr_idx, _ = folds[-1]
    Xtr, ytr = X.iloc[tr_idx], y.iloc[tr_idx]

    rf = RandomForestRegressor(
        n_estimators=600,
        max_depth=None,
        n_jobs=-1,
        random_state=SEED,
        oob_score=False,
    )
    boruta = BorutaPy(
        estimator=rf,
        n_estimators="auto",
        alpha=0.05,
        two_step=True,
        max_iter=100,
        random_state=SEED
    )
    try:
        boruta.fit(Xtr.fillna(0).values, ytr.values)
        status = boruta.support_.astype(int) + 0.5 * boruta.support_weak_.astype(int)
        return pd.Series(status, index=X.columns).sort_values(ascending=False)
    except Exception as e:
        print(f"[WARN] Boruta falló: {e}")
        return pd.Series(0.0, index=X.columns)


### C-Serie: Filtros de validación  <a id='c-serie'></a>

**Alcance**: Se aplican a todas las *features* **no mandatorias**.

- **C1** (*missing values*): se dropean *features* con `frac_missing > 1%`.
- **C2** (*low variance / constante*): se dropean *features* donde un solo valor domina `≥ 99%` de las filas.
- **C3** (*clustering por correlación*): se forman *clusters* con `|corr| ≥ 0.95` y se retienen **Top-2** de cada cluster
  según *Mutual Information* promedio en CV (Cálculo de MI sólo con *folds de entrenamiento*).


In [6]:
# C1: Missing values 
if MANDATORY:
    feat_cols = [c for c in X.columns if not is_mandatory(c)]  # sólo procesables
else:
    feat_cols = X.columns.tolist()  # todos

miss_frac = X[feat_cols].isna().mean().sort_values(ascending=False)
drop_c1 = miss_frac[miss_frac > MAX_MISSING_FRAC].index.tolist()

# C2: Low variance (valor dominante)
def dominant_ratio(s: pd.Series) -> float:
    if s.empty:
        return 0.0
    vc = s.value_counts(dropna=False)
    if len(vc) == 0:
        return 0.0
    return float(vc.iloc[0]) / float(len(s))

dom_ratios = X[feat_cols].apply(dominant_ratio)
drop_c2 = dom_ratios[dom_ratios >= CONST_DOMINANCE_THR].index.tolist()

# Aplicar drops C1 y C2
drop_c12 = sorted(set(drop_c1 + drop_c2))
if MANDATORY:
    keep_after_c12 = [c for c in X.columns if c not in drop_c12 or is_mandatory(c)]
else:
    keep_after_c12 = [c for c in X.columns if c not in drop_c12]
X_c12 = X[keep_after_c12].copy()

_stamp(f"C1 drop: {len(drop_c1)} | C2 drop: {len(drop_c2)} | Total únicos C1–C2: {len(drop_c12)}")
_stamp(f"Shape tras C1–C2: {X_c12.shape}")

# C3: Correlation clustering + MI Top-2
# Matriz de correlación en todo el conjunto (no usa target => sin leakage)
corr = X_c12.select_dtypes(include=[np.number]).corr().abs()

# Construir clusters por umbral
visited = set()
clusters = []
cols_numeric = corr.columns.tolist()

for col in cols_numeric:
    if col in visited:
        continue
    # cluster = todos los nodos conectados con |corr| >= thr
    related = set([col])
    frontier = [col]
    while frontier:
        cur = frontier.pop()
        visited.add(cur)
        neighbors = corr.index[(corr[cur] >= CORR_CLUSTER_THR) & (corr.index != cur)].tolist()
        for nb in neighbors:
            if nb not in related:
                related.add(nb)
                if nb not in visited:
                    frontier.append(nb)
    clusters.append(sorted(list(related)))

# Calcular MI promedio por fold sólo sobre features candidatas (excluye mandatorias)
mi_scores = cv_mutual_info(X_c12[cols_numeric], y)

# En cada cluster, seleccionar Top-2 por MI (si hay menos, se retienen todas)
selected_c3 = set()
for cl in clusters:
    cl_sorted = sorted(cl, key=lambda c: mi_scores.get(c, 0.0), reverse=True)
    top2 = cl_sorted[:2]
    for c in top2:
        selected_c3.add(c)

# Componer columnas finales tras C3
if MANDATORY:
    keep_c3 = sorted(set([c for c in X_c12.columns if is_mandatory(c)]) | selected_c3 | set([c for c in X_c12.columns if c not in cols_numeric]))
else:
    keep_c3 = sorted(set(selected_c3) | set([c for c in X_c12.columns if c not in cols_numeric]))
X_c3 = X_c12[keep_c3].copy()
_stamp(f"Clusters C3: {len(clusters)} | Seleccionadas por C3: {len(selected_c3)} | Shape X_c3: {X_c3.shape}")


[19:47:38] C1 drop: 0 | C2 drop: 0 | Total únicos C1–C2: 0
[19:47:38] Shape tras C1–C2: (644, 354)
[19:47:40] Clusters C3: 214 | Seleccionadas por C3: 253 | Shape X_c3: (644, 253)


### Proceso en paralelo: Relevancias  <a id='paralelo-relevancias'></a>

Se calculan **tres** rankings independientes sobre `X_c3`:

- **C4 – XGBoost**: promedio de importancias en *folds*.
- **C5 – Boruta (RF)**: *selected=1*, *weak=0.5*, *rejected=0* sobre el último *fold* de entrenamiento.
- **C6 – SHAP (XGB)**: media de `|SHAP|` en muestra del set de entrenamiento por *fold*.

> Nota: si alguna librería no está instalada, el ranking correspondiente será cero para todas las variables.


In [7]:
# Columnas numéricas (XGBoost/SHAP requieren numérico)
numeric_cols = X_c3.select_dtypes(include=[np.number]).columns.tolist()
Xc3_num = X_c3[numeric_cols].copy()

# C4: XGBoost
xgb_rank = cv_xgb_importance(Xc3_num, y).rename("xgb_importance")

# C5: Boruta
boruta_rank = boruta_selection(Xc3_num, y).rename("boruta_score")

# C6: SHAP
shap_rank = fold_shap_importance(Xc3_num, y).rename("shap_mean_abs")

# Ensamble de rankings en un único DataFrame
rank_df = pd.concat([xgb_rank, boruta_rank, shap_rank], axis=1).fillna(0.0)
rank_df["avg_norm_rank"] = 0.0

# Normalización por columna (min-max) antes de promediar
for col in ["xgb_importance", "boruta_score", "shap_mean_abs"]:
    vals = rank_df[col].values
    vmin, vmax = np.nanmin(vals), np.nanmax(vals)
    if vmax > vmin:
        rank_df[col + "_norm"] = (vals - vmin) / (vmax - vmin)
    else:
        rank_df[col + "_norm"] = 0.0
rank_df["avg_norm_rank"] = rank_df[[c for c in rank_df.columns if c.endswith("_norm")]].mean(axis=1)

rank_df = rank_df.sort_values("avg_norm_rank", ascending=False)
_stamp(f"Ranking combinado (shape={rank_df.shape})")
rank_df.head(10)

[19:48:50] Ranking combinado (shape=(253, 7))


,xgb_importance,boruta_score,shap_mean_abs,avg_norm_rank,xgb_importance_norm,boruta_score_norm,shap_mean_abs_norm
merval_cierre_ma_3,0.141575,1.0,166700.896289,1.000000,1.000000,1.0,1.000000
merval_apertura,0.120840,1.0,115954.269141,0.849707,0.853538,1.0,0.695583
emae_tendencia_ciclo_ma_5,0.132671,1.0,12223.472729,0.670145,0.937108,1.0,0.073326
badlar_ma_3,0.110678,1.0,20699.915527,0.635312,0.781763,1.0,0.124174
emae_desestacionalizado_lag2,0.113965,1.0,3400.900121,0.608460,0.804978,1.0,0.020401
emae_tendencia_ciclo_lag3,0.109259,1.0,3706.422852,0.597990,0.771737,1.0,0.022234
emae_desestacionalizado_lag3,0.029558,1.0,1798.404987,0.406524,0.208783,1.0,0.010788
badlar_ma_5,0.020842,1.0,6334.535132,0.395072,0.147218,1.0,0.037999
emae_tendencia_ciclo_hp_ema20,0.091637,0.5,3629.041168,0.389679,0.647266,0.5,0.021770
base_monetaria_std20,0.010320,1.0,2700.010310,0.363030,0.072894,1.0,0.016197


### Selecciones Top-K y Top-60 final  <a id='selecciones'></a>

- **D**: Selección **Top-20** de cada método (XGB, Boruta, SHAP).  
- **E**: Completar **Top-60** tomando el *promedio normalizado* (`avg_norm_rank`).  
- Si MANDATORY, se preservan todas las *features mandatorias* (`merval_*`).  

In [8]:
# D) Top-20 por método
top20_xgb    = rank_df.sort_values("xgb_importance", ascending=False).head(TOPK_PER_METHOD).index.tolist()
top20_boruta = rank_df.sort_values("boruta_score",   ascending=False).head(TOPK_PER_METHOD).index.tolist()
top20_shap   = rank_df.sort_values("shap_mean_abs",  ascending=False).head(TOPK_PER_METHOD).index.tolist()

selected_d = list(dict.fromkeys(top20_xgb + top20_boruta + top20_shap))  # únicos manteniendo orden

# E) Completar Top-60 con promedio normalizado
if MANDATORY:
    mandatory_cols = [c for c in X_c3.columns if is_mandatory(c)]
else:
    mandatory_cols = []  # IGNORE
pool = [c for c in rank_df.index if c not in selected_d]
need_more = max(0, 60 - (len(selected_d) + len(mandatory_cols)))

extra = rank_df.loc[pool].sort_values("avg_norm_rank", ascending=False).head(need_more).index.tolist()

final_top60 = list(dict.fromkeys(mandatory_cols + selected_d + extra))[:60]

_stamp(f"Mandatorias: {len(mandatory_cols)} | D únicos: {len(selected_d)} | Extra: {len(extra)} | Top-60 final: {len(final_top60)}")

# Conjunto final de entrenamiento
cols_final = final_top60 + [c for c in X_c3.columns if is_mandatory(c) and c not in final_top60]
cols_final = list(dict.fromkeys(cols_final))  # por si acaso
X_final = X_c3[final_top60].copy()

_stamp(f"X_final shape: {X_final.shape}")
pd.DataFrame({"feature": final_top60}).head(10)


[19:48:50] Mandatorias: 0 | D únicos: 53 | Extra: 7 | Top-60 final: 60
[19:48:50] X_final shape: (644, 60)


,feature
0,merval_cierre_ma_3
1,emae_tendencia_ciclo_ma_5
2,merval_apertura
3,emae_desestacionalizado_lag2
4,badlar_ma_3
5,emae_tendencia_ciclo_lag3
6,emae_tendencia_ciclo_hp_ema20
7,inflacion_lag2
8,emae_desestacionalizado_lag3
9,badlar_ma_5


In [9]:
rank_df.head(30)

,xgb_importance,boruta_score,shap_mean_abs,avg_norm_rank,xgb_importance_norm,boruta_score_norm,shap_mean_abs_norm
merval_cierre_ma_3,0.141575,1.0,166700.896289,1.000000,1.000000,1.0,1.000000
merval_apertura,0.120840,1.0,115954.269141,0.849707,0.853538,1.0,0.695583
emae_tendencia_ciclo_ma_5,0.132671,1.0,12223.472729,0.670145,0.937108,1.0,0.073326
badlar_ma_3,0.110678,1.0,20699.915527,0.635312,0.781763,1.0,0.124174
emae_desestacionalizado_lag2,0.113965,1.0,3400.900121,0.608460,0.804978,1.0,0.020401
emae_tendencia_ciclo_lag3,0.109259,1.0,3706.422852,0.597990,0.771737,1.0,0.022234
emae_desestacionalizado_lag3,0.029558,1.0,1798.404987,0.406524,0.208783,1.0,0.010788
badlar_ma_5,0.020842,1.0,6334.535132,0.395072,0.147218,1.0,0.037999
emae_tendencia_ciclo_hp_ema20,0.091637,0.5,3629.041168,0.389679,0.647266,0.5,0.021770
base_monetaria_std20,0.010320,1.0,2700.010310,0.363030,0.072894,1.0,0.016197


In [10]:
final_top60

['merval_cierre_ma_3',
 'emae_tendencia_ciclo_ma_5',
 'merval_apertura',
 'emae_desestacionalizado_lag2',
 'badlar_ma_3',
 'emae_tendencia_ciclo_lag3',
 'emae_tendencia_ciclo_hp_ema20',
 'inflacion_lag2',
 'emae_desestacionalizado_lag3',
 'badlar_ma_5',
 'base_monetaria_std20',
 'tasa_prestamos_personales_ma_5',
 'emae_desestacionalizado_hp_ema20',
 'tc_mayorista_std20',
 'emae_desestacionalizado_diff10',
 'tasa_prestamos_personales_ma_3',
 'merval_maximo_hp_rm20',
 'emae_desestacionalizado_hp_rm20',
 'emae_desestacionalizado_std20',
 'emae_original_std20',
 'tasa_prestamos_personales_std20',
 'm1_std20',
 'merval_minimo_spec_centroid',
 'tc_minorista_std20',
 'm2_transaccional_std20',
 'emae_tendencia_ciclo_diff10',
 'inflacion_diff10',
 'merval_maximo_diff10',
 'merval_cierre_hp_rm20',
 'badlar_spec_centroid',
 'badlar_std20',
 'inflacion_hp_ema20',
 'merval_cierre_diff3',
 'merval_minimo_pct1',
 'inflacion_hp_rm20',
 'badlar_hp_ema20',
 'merval_cierre_pct3',
 'merval_cierre_std20',


### Export de artefactos  <a id='export'></a>

Se exportan:

- `selected_features_top60.csv`: lista ordenada de *features*.
- `feature_ranking.csv`: tabla de *rankings* con normalizados y promedio.
- `dataset_top60.parquet`: dataset reducido a Top-60 + *target*.
- `feature_selection_report.md`: breve reporte con conteos y umbrales utilizados.


In [11]:
pd.concat([X_final, y.rename(TARGET_COL)], axis=1)

,merval_cierre_ma_3,emae_tendencia_ciclo_ma_5,merval_apertura,emae_desestacionalizado_lag2,badlar_ma_3,emae_tendencia_ciclo_lag3,emae_tendencia_ciclo_hp_ema20,inflacion_lag2,emae_desestacionalizado_lag3,badlar_ma_5,...,merval_cierre_pct1,merval_maximo_diff1,m2_std20,badlar_diff10,m2_transaccional_diff10,m2_pct10,emae_tendencia_ciclo_diff3,tc_mayorista_pct10,m2_spec_centroid,merval_apertura_+1
fecha,,,,,,,,,,,,,,,,,,,,,
2023-01-02,2.028334e+05,149.643342,2.020851e+05,148.762438,68.395833,149.717452,-0.335258,5.1,148.762438,68.3625,...,0.024590,5181.0,6.781839e+05,-1.0625,856731.367,0.085630,-0.370548,0.025857,0.085979,2.070543e+05
2023-01-03,2.025000e+05,149.569232,2.070543e+05,148.762438,68.375000,149.717452,-0.303329,5.1,148.762438,68.4375,...,-0.041988,-1155.0,6.646710e+05,0.1250,589102.429,0.045979,-0.370548,0.025872,0.080954,1.983605e+05
2023-01-04,2.021585e+05,149.495123,1.983605e+05,149.687389,68.500000,149.717452,-0.274440,6.0,148.762438,68.5875,...,0.013612,-5615.0,6.386205e+05,1.2500,574635.280,0.042834,-0.370548,0.025251,0.079350,2.013798e+05
2023-01-05,2.030224e+05,149.421013,2.013798e+05,149.687389,68.791667,149.346903,-0.248303,6.0,149.687389,68.6250,...,0.042701,8613.0,6.038151e+05,-0.7500,419228.439,0.029672,0.000000,0.024916,0.074886,2.137942e+05
2023-01-06,2.067842e+05,149.346903,2.137942e+05,149.687389,69.000000,149.346903,-0.224655,6.0,149.687389,68.6250,...,0.000000,6228.0,5.691356e+05,0.5625,394035.393,0.001887,0.000000,0.025107,0.074847,2.137942e+05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-08-22,2.097105e+06,151.501857,2.103163e+06,151.449169,51.333333,151.501857,-0.028904,1.9,151.449169,51.2250,...,0.001444,24260.0,1.850387e+06,12.8750,-4831754.290,-0.032333,0.000000,-0.007137,0.071934,2.106200e+06
2025-08-25,2.077072e+06,151.501857,2.106200e+06,151.449169,50.270833,151.501857,-0.026151,1.9,151.449169,51.5750,...,-0.040047,-47310.0,1.846232e+06,9.3750,-3691663.660,0.006069,0.000000,0.011095,0.072333,2.021852e+06
2025-08-26,2.053799e+06,151.501857,2.021852e+06,151.449169,52.125000,151.501857,-0.023661,1.9,151.449169,51.9875,...,0.005685,-51850.0,1.859643e+06,16.6250,-3187575.720,0.007838,0.000000,0.016407,0.076821,2.033345e+06


In [13]:
# Archivos de salida
features_csv = f"{DATASET_NAME}_selected_features_top60.csv"
ranking_csv  = f"{DATASET_NAME}_feature_ranking.csv"
dataset_csv   = f"../inputs/{DATASET_NAME}_fs_top60.csv"
report_md    = f"{DATASET_NAME}_feature_selection_report.md"

# Guardar
pd.DataFrame({"feature": final_top60}).to_csv(features_csv, index=False)
rank_df.to_csv(ranking_csv, index=True)
pd.concat([X_final, y.rename(TARGET_COL)], axis=1).to_csv(dataset_csv)

# Reporte rápido
report = f"""
# Feature Selection Report
Fecha: {datetime.now():%Y-%m-%d %H:%M:%S}

- Input: `{INPUT_DATA_FILE}`
- Filtrado C1 (missing > {MAX_MISSING_FRAC:.2%}): {len(drop_c1)}
- Filtrado C2 (dominancia >= {CONST_DOMINANCE_THR:.2%}): {len(drop_c2)}
- Clusters C3 (|corr| >= {CORR_CLUSTER_THR:.0%}): {len(clusters)}
- Mandatorias retenidas: {len(mandatory_cols)}
- Top-20 por método: XGB={len(top20_xgb)}, Boruta={len(top20_boruta)}, SHAP={len(top20_shap)}
- Top-60 final: {len(final_top60)}

Parámetros:
- N_SPLITS={N_SPLITS}
- TARGET={TARGET_COL}
"""

with open(report_md, "w", encoding="utf-8") as f:
    f.write(report)

_stamp(f"Exportado:\n - {features_csv}\n - {ranking_csv}\n - {dataset_csv}\n - {report_md}")


[19:54:01] Exportado:
 - dataset_v2_selected_features_top60.csv
 - dataset_v2_feature_ranking.csv
 - ../inputs/dataset_v2_fs_top60.csv
 - dataset_v2_feature_selection_report.md
